# IAM Handwriting Dataset Training with CUT Model + OCR Loss

This notebook trains a **Contrastive Unpaired Translation (CUT)** model on the IAM handwriting dataset with an **OCR loss** to preserve text content during style transfer.

## What this notebook does:
1. ✅ Clones your Modified CycleGAN repository
2. ✅ Installs dependencies (including EasyOCR)
3. ✅ Verifies the dataset and OCR module structure
4. ✅ Adds `--lambda_OCR` argument to training options
5. ✅ Trains the model with OCR loss enabled
6. ✅ Checks training results

## Requirements:
- IAM dataset should be in Kaggle input at: `/kaggle/input/iam-dataset-cleaned/iam_cyclegan_dataset_clean_1024x256`
- Dataset structure should have `trainA/` and `trainB/` folders
- Your GitHub repo should include the OCR module at `models/my_ocr/ocr_module.py`

---


In [ ]:
# 1️⃣ Clone your updated repo into /kaggle/working/CUT
!git clone https://github.com/Devyansh99/Modified_cyclegan.git /kaggle/working/CUT

# 2️⃣ Move into the repo directory
%cd /kaggle/working/CUT

# 3️⃣ Install dependencies
!pip install -r requirements.txt --quiet

# 4️⃣ Install EasyOCR for OCR functionality
!pip install easyocr --quiet

print("✅ Setup complete!")


Cloning into '/kaggle/working/CUT'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 63 (delta 2), reused 63 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 15.28 MiB | 19.01 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/kaggle/working/CUT
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.0 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os

# Verify the CUT directory structure
cut_dir = "/kaggle/working/CUT"
train_file = os.path.join(cut_dir, "train.py")

assert os.path.exists(train_file), "❌ train.py not found!"
print("✅ Found train.py at:", train_file)

# Verify OCR module exists
ocr_module_dir = os.path.join(cut_dir, "models", "my_ocr")
ocr_module_file = os.path.join(ocr_module_dir, "ocr_module.py")

if os.path.exists(ocr_module_file):
    print("✅ OCR module found at:", ocr_module_file)
else:
    print("❌ OCR module not found! Please ensure models/my_ocr/ocr_module.py exists in your GitHub repo")

# Verify dataset structure
dataset_path = "/kaggle/input/iam-dataset-cleaned/iam_cyclegan_dataset_clean_1024x256"
if os.path.exists(dataset_path):
    print("✅ Dataset found at:", dataset_path)
    # Check for trainA and trainB folders
    trainA = os.path.join(dataset_path, "trainA")
    trainB = os.path.join(dataset_path, "trainB")
    if os.path.exists(trainA):
        print(f"  ✅ trainA folder found with {len(os.listdir(trainA))} images")
    else:
        print("  ⚠️  trainA folder not found!")
    if os.path.exists(trainB):
        print(f"  ✅ trainB folder found with {len(os.listdir(trainB))} images")
    else:
        print("  ⚠️  trainB folder not found!")
else:
    print("⚠️  Dataset not found. Adjust the path in the training command.")


✅ Found train.py at: /kaggle/working/CUT/train.py
✅ OCR module found at: /kaggle/working/CUT/models/my_ocr


In [ ]:
import os

# Add --lambda_OCR argument to train_options.py
file_path = "/kaggle/working/CUT/options/train_options.py"

# Read the file
with open(file_path, 'r') as f:
    content = f.read()

# Check if lambda_OCR already exists
if '--lambda_OCR' in content:
    print("ℹ️  --lambda_OCR argument already exists in train_options.py")
else:
    # Find the line with "self.isTrain = True" and add the argument before it
    lines = content.split('\n')
    new_lines = []
    
    for line in lines:
        if 'self.isTrain = True' in line:
            # Add the lambda_OCR argument before self.isTrain
            new_lines.append("        parser.add_argument('--lambda_OCR', type=float, default=0.1, help='weight for OCR loss (set 0 to disable)')")
            new_lines.append("")
        new_lines.append(line)
    
    # Write the modified content back
    with open(file_path, 'w') as f:
        f.write('\n'.join(new_lines))
    
    print("✅ Added --lambda_OCR argument to train_options.py")

# Verify the change
with open(file_path, 'r') as f:
    if '--lambda_OCR' in f.read():
        print("✅ Verification passed: --lambda_OCR is now in train_options.py")


✅ Added --lambda_OCR argument to train_options.py


In [ ]:
# Train the model with OCR loss
!python /kaggle/working/CUT/train.py \
  --dataroot /kaggle/input/iam-dataset-cleaned/iam_cyclegan_dataset_clean_1024x256 \
  --name cut_handwriting_ocr \
  --model cut \
  --dataset_mode unaligned \
  --n_epochs 10 \
  --n_epochs_decay 10 \
  --batch_size 1 \
  --load_size 286 \
  --crop_size 256 \
  --gpu_ids 0 \
  --lambda_OCR 0.1 \
  --display_id 0 \
  --print_freq 50 \
  --save_epoch_freq 5 \
  --save_latest_freq 1000


Traceback (most recent call last):
  File "/kaggle/working/CUT/train.py", line 3, in <module>
    from options.train_options import TrainOptions
  File "/kaggle/working/CUT/options/train_options.py", line 47, in <module>
    parser.add_argument('--lambda_OCR', type=float, default=0.0, help='weight for OCR loss (set 0 to disable OCR loss)')
    ^^^^^^
NameError: name 'parser' is not defined


In [ ]:
# Check training results
import os

checkpoints_dir = "/kaggle/working/CUT/checkpoints/cut_handwriting_ocr"
if os.path.exists(checkpoints_dir):
    print("✅ Training checkpoints saved at:", checkpoints_dir)
    files = os.listdir(checkpoints_dir)
    print(f"📁 Found {len(files)} files/folders")
    for f in sorted(files)[:10]:  # Show first 10
        print(f"  - {f}")
else:
    print("⚠️  No checkpoints found yet. Training may still be in progress or failed.")


## Optional: Test the trained model

Run inference on test images after training completes.


---

## ⚠️ Important: Update Your GitHub Repository

**Before running this notebook again**, you need to push the fixed code to your GitHub repository!

The OCR loss has been fixed to use a **differentiable L1 loss** instead of non-differentiable text similarity. This ensures gradients flow properly through the generator.

### Files that were modified (commit these to GitHub):
1. `train.py` - Simplified OCR loss integration
2. `models/cut_model.py` - Integrated OCR loss into `compute_G_loss()`
3. `models/my_ocr/ocr_module.py` - Changed to use L1 loss (differentiable)

### To update your repo:
```bash
git add train.py models/cut_model.py models/my_ocr/ocr_module.py
git commit -m "Fixed OCR loss to use differentiable L1 loss"
git push origin master
```

Then re-run this notebook from the beginning!

---


In [ ]:
# Test the trained model (optional - run after training completes)
# Uncomment and adjust paths as needed

# !python /kaggle/working/CUT/test.py \
#   --dataroot /kaggle/input/iam-dataset-cleaned/iam_cyclegan_dataset_clean_1024x256/testA \
#   --name cut_handwriting_ocr \
#   --model cut \
#   --phase test \
#   --epoch latest \
#   --num_test 50
